In [1]:
"""
B1. Test harness for the signed-magnitude CORDIC-based arccos quantum circuit
-------------------------------------------------------------------------------
This module evaluates a quantum circuit implementation of arccos(x) based on a
signed-magnitude fixed-point encoding and a CORDIC-style construction.

For each specified real input x ∈ [-1, 1], the workflow proceeds as follows:
  1) Quantize x to the nearest representable fixed-point value x_quant
     determined by the magnitude register size.
  2) Prepare the signed-magnitude computational basis state
     (sign qubit + magnitude register).
  3) Apply the arccos quantum circuit to produce the output angle register.
  4) Simulate the circuit using Qiskit Aer and measure the theta register.
  5) Compute summary statistics (mean, standard deviation, mode, and absolute
     error) relative to the exact classical value arccos(x_quant).

The script prints a formatted results table, reports aggregate error metrics,
and returns structured data suitable for further analysis or visualization.
"""

import _init_path
from pathlib import Path

from __future__ import annotations

import time
from typing import List, Dict, Any
from math import acos, pi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qiskit import ClassicalRegister, QuantumRegister, QuantumCircuit, transpile
from qiskit_aer import AerSimulator

from quantum.arccos_cordic import build_arccos_signedmag_circuit
from utils import _apply_publication_rcparams, _force_opaque

Repo root: /Users/junaida/Documents/Non-Uniform-QFT
Added to sys.path: /Users/junaida/Documents/Non-Uniform-QFT/src


In [2]:
def _fmt_float(x: Any, ndigits: int = 10) -> str:
    """Format a value as a floating-point string for tabular display.

    If the input is None, an empty string is returned. If the input cannot be
    converted to float, its string representation is returned.

    Args:
        x: Value to format (typically a float-like value or None).
        ndigits: Number of digits after the decimal point.

    Returns:
        A formatted string suitable for table display.

    Raises:
        ValueError: If ndigits < 0.
    """
    if ndigits < 0:
        raise ValueError("ndigits must be >= 0.")

    if x is None:
        return ""
    try:
        return f"{float(x): .{ndigits}f}"
    except Exception:
        return str(x)


def print_arccos_results_table(rows: List[dict]) -> None:
    """Print a box-style ASCII table summarizing arccos test results.

    This routine formats a list of per-input result dictionaries into a boxed
    ASCII table suitable for terminal display.

    Args:
        rows: List of dictionaries containing per-input statistics and errors.

    Returns:
        None.

    Raises:
        ValueError: If rows is not a list of dictionaries.
    """
    if not isinstance(rows, list):
        raise ValueError("rows must be a list of dictionaries.")

    headers = [
        "#",
        "x_req",
        "x_quant",
        "acos(xq)",
        "mean(theta)",
        "abs_err",
        "std(theta)",
        "mode(theta)",
    ]
    aligns = ["R"] * len(headers)

    table_rows: List[List[str]] = []
    for i, r in enumerate(rows, 1):
        table_rows.append(
            [
                str(i),
                _fmt_float(r.get("x_req"), 6),
                _fmt_float(r.get("x_quant"), 6),
                _fmt_float(r.get("acos_exact"), 10),
                _fmt_float(r.get("mean_theta"), 10),
                _fmt_float(r.get("abs_err"), 10),
                _fmt_float(r.get("std_theta"), 10),
                _fmt_float(r.get("mode_theta"), 10),
            ]
        )

    widths = [len(h) for h in headers]
    for row in table_rows:
        for j, cell in enumerate(row):
            widths[j] = max(widths[j], len(cell))

    TL, TR, BL, BR = "┌", "┐", "└", "┘"
    H, V = "─", "│"
    TJ, BJ, LJ, RJ, CJ = "┬", "┴", "├", "┤", "┼"

    def hline(left: str, mid: str, right: str) -> str:
        return left + mid.join(H * (w + 2) for w in widths) + right

    def format_cell(text: str, w: int, align: str, header: bool = False) -> str:
        if header:
            return f" {text:^{w}} "
        if align == "R":
            return f" {text:>{w}} "
        return f" {text:<{w}} "

    def render_row(row: List[str], header: bool = False) -> str:
        return V + V.join(
            format_cell(row[j], widths[j], aligns[j], header=header)
            for j in range(len(headers))
        ) + V

    print(hline(TL, TJ, TR))
    print(render_row(headers, header=True))
    print(hline(LJ, CJ, RJ))
    for row in table_rows:
        print(render_row(row))
    print(hline(BL, BJ, BR))


def print_arccos_summary(rows: List[dict]) -> None:
    """Print absolute-error summary statistics over result rows."""
    if not isinstance(rows, list):
        raise ValueError("rows must be a list of dictionaries.")

    if not rows:
        return
    abs_errs = [float(r["abs_err"]) for r in rows if r.get("abs_err") is not None]
    if not abs_errs:
        return

    mean_abs = float(np.mean(abs_errs))
    max_abs = float(np.max(abs_errs))
    print("\nSummary:")
    print(f"Mean abs error: {mean_abs:.10f}")
    print(f"Max abs error: {max_abs:.10f}")


def _encode_signedmag_fixed_point(x: float, m: int) -> Tuple[int, int, float]:
    """Encode x in [-1, 1] using the signed-magnitude fixed-point scheme."""
    if m < 1:
        raise ValueError("m must be >= 1.")
    if not (-1.0 <= x <= 1.0):
        raise ValueError("x must be in [-1, 1].")

    sign_bit = 1 if x < 0 else 0
    scale = 2 ** (m - 1)

    u = int(np.round(min(abs(x), (scale - 1) / scale) * scale))
    u = max(0, min(u, scale - 1))

    x_quant = (-1 if sign_bit else 1) * (u / scale)
    return sign_bit, u, x_quant


def _prepare_input_on_regs(
    qc,
    magReg,
    signQ,
    sign_bit: int,
    u: int,
) -> None:
    """Prepare a signed-magnitude basis state using X gates."""
    if sign_bit not in (0, 1):
        raise ValueError("sign_bit must be 0 or 1.")
    if u < 0:
        raise ValueError("u must be >= 0.")
    if u >= (1 << len(magReg)):
        raise ValueError("u does not fit in the provided magReg.")

    if sign_bit == 1:
        qc.x(signQ[0])

    for i in range(len(magReg)):
        if (u >> i) & 1:
            qc.x(magReg[i])


def _counts_to_stats(counts: dict, p: int) -> Dict[str, Any]:
    """Convert theta measurement counts into basic statistics."""
    if p < 0:
        raise ValueError("p must be >= 0.")
    if not counts:
        raise ValueError("counts must be non-empty.")

    shots = sum(counts.values())
    if shots <= 0:
        raise ValueError("Total shots must be > 0.")

    theta_int_vals = np.array([int(bitstr, 2) for bitstr in counts.keys()], dtype=np.int64)
    probs = np.array([counts[b] / shots for b in counts.keys()], dtype=np.float64)

    mean_int = float(np.sum(theta_int_vals * probs))
    mean = mean_int / (2 ** p)

    var_int = float(np.sum(((theta_int_vals - mean_int) ** 2) * probs))
    std = np.sqrt(var_int) / (2 ** p)

    mode_bitstr = max(counts.items(), key=lambda kv: kv[1])[0]
    mode_int = int(mode_bitstr, 2)
    mode = mode_int / (2 ** p)

    return {
        "shots": shots,
        "mean_theta": mean,
        "std_theta": std,
        "mode_theta": mode,
        "mode_int": mode_int,
    }


def test_arccos_circuit(
    m: int,
    p: int,
    xs: Iterable[float],
    shots: int = 2048,
    clean_cordic_ancillas: bool = True,
    seed: int = 1234,
) -> List[Dict[str, Any]]:
    """Build and simulate arccos circuits for a collection of inputs."""
    if m < 1:
        raise ValueError("m must be >= 1.")
    if p < 0:
        raise ValueError("p must be >= 0.")
    if shots < 1:
        raise ValueError("shots must be >= 1.")

    # Provided by your codebase:
    # base = build_arccos_signedmag_circuit(m, p, clean_cordic_ancillas=clean_cordic_ancillas)
    base = build_arccos_signedmag_circuit(m, p, clean_cordic_ancillas=clean_cordic_ancillas)

    regs = {r.name: r for r in base.qregs}
    magReg = regs["mag"]
    signQ = regs["sgn"]
    thetaReg = regs["theta"]
    w = len(thetaReg)

    backend = AerSimulator(method="matrix_product_state")

    circuits: List[Any] = []
    meta: List[Tuple[float, float, int, int]] = []
    for x in xs:
        sign_bit, u, x_quant = _encode_signedmag_fixed_point(float(x), m)

        ctheta = ClassicalRegister(w, "ctheta")
        qc = QuantumCircuit(*base.qregs, ctheta)

        _prepare_input_on_regs(qc, magReg, signQ, sign_bit, u)
        qc.compose(base, inplace=True)
        qc.measure(thetaReg, ctheta)

        circuits.append(qc)
        meta.append((float(x), float(x_quant), int(sign_bit), int(u)))

    tqcs = transpile(circuits, backend, optimization_level=1, seed_transpiler=seed)
    result = backend.run(tqcs, shots=shots).result()

    rows: List[Dict[str, Any]] = []
    for i, (x_req, xq, sign_bit, u) in enumerate(meta):
        counts = result.get_counts(i)
        stats = _counts_to_stats(counts, p)

        exact = float(np.arccos(xq))
        mean_theta = float(stats["mean_theta"])
        abs_err = abs(mean_theta - exact)

        rows.append(
            {
                "x_req": x_req,
                "x_quant": xq,
                "acos_exact": exact,
                "mean_theta": mean_theta,
                "abs_err": abs_err,
                "std_theta": float(stats["std_theta"]),
                "mode_theta": float(stats["mode_theta"]),
            }
        )

    print_arccos_results_table(rows)
    print_arccos_summary(rows)
    return rows


def _coerce_rows_for_plotting(rows: Any) -> List[Dict[str, Any]]:
    """Coerce result rows into the key schema expected by the plotter."""
    if not rows:
        return []

    if isinstance(rows, list) and isinstance(rows[0], dict):
        needed = {"x_req", "x_quant", "acos_xq", "mean_theta"}
        if needed.issubset(set(rows[0].keys())):
            return rows

        mapped: List[Dict[str, Any]] = []
        for r in rows:
            mapped.append(
                {
                    "x_req": r.get("x_req", r.get("x", None)),
                    "x_quant": r.get("x_quant", r.get("xq", r.get("x_hat", None))),
                    "acos_xq": r.get("acos_xq", r.get("acos(xq)", r.get("acos_x_quant", None))),
                    "mean_theta": r.get("mean_theta", r.get("mean(theta)", r.get("theta_mean", None))),
                    "std_theta": r.get("std_theta", r.get("std(theta)", r.get("theta_std", None))),
                }
            )

        cleaned = [
            rr
            for rr in mapped
            if rr["x_req"] is not None
            and rr["x_quant"] is not None
            and rr["acos_xq"] is not None
            and rr["mean_theta"] is not None
        ]
        return cleaned

    raise TypeError("Unexpected `rows` format; expected list[dict].")


def _ensure_subdir_under_existing_figures(out_dir: Path) -> Path:
    """Ensure only the subdirectory is created under an existing base figures dir.

    Contract:
      - The parent directory of out_dir must already exist.
        Example: out_dir = figures/B1_arccos  =>  'figures' must exist already.
      - This function creates only out_dir if missing.

    Raises:
      - FileNotFoundError if the parent directory does not exist.
      - NotADirectoryError if parent exists but is not a directory.
    """
    out_dir = Path(out_dir)
    base_dir = out_dir.parent

    if not base_dir.exists():
        raise FileNotFoundError(
            f"Base figures directory does not exist: {base_dir}. "
            f"Create it first (e.g., mkdir -p {base_dir})."
        )
    if not base_dir.is_dir():
        raise NotADirectoryError(f"Base figures path is not a directory: {base_dir}")

    out_dir.mkdir(exist_ok=True)
    return out_dir


def plot_arccos_actual_vs_predicted(
    rows: List[Dict[str, Any]],
    p: int,
    out_dir: Path,
    show_figures: bool = False,
    use_xhat_on_xaxis: bool = False,
) -> None:
    r"""Plot actual versus predicted arccos values for a fixed p.

    "Actual"    := arccos(x_quant) computed classically from the quantized input
                   (stored as rows[i]["acos_exact"]).
    "Predicted" := mean(theta) estimated from circuit measurements
                   (stored as rows[i]["mean_theta"]).
    """
    import matplotlib.pyplot as plt

    if not rows:
        return

    _apply_publication_rcparams()

    # IMPORTANT CHANGE:
    # Do not create 'figures/' automatically; only create the subfolder, e.g. figures/B1_arccos
    out_dir = _ensure_subdir_under_existing_figures(Path(out_dir))

    x = np.asarray([r["x_quant"] if use_xhat_on_xaxis else r["x_req"] for r in rows], dtype=float)
    y_true = np.asarray([r["acos_exact"] for r in rows], dtype=float)   # arccos(x_quant)
    y_pred = np.asarray([r["mean_theta"] for r in rows], dtype=float)   # predicted mean(theta)

    y_std: Optional[np.ndarray] = None
    if rows[0].get("std_theta", None) is not None:
        y_std = np.asarray([r["std_theta"] for r in rows], dtype=float)

    order = np.argsort(x)
    x = x[order]
    y_true = y_true[order]
    y_pred = y_pred[order]
    if y_std is not None:
        y_std = y_std[order]

    fig, ax = plt.subplots(figsize=(6.6, 4.3))

    ax.plot(x, y_true, marker="o", label=r"Actual")
    ax.plot(x, y_pred, marker="s", label=r"Predicted")

    if y_std is not None:
        ax.fill_between(x, y_pred - y_std, y_pred + y_std, alpha=0.10)

    ax.set_xlabel(r"$\widehat{x}$" if use_xhat_on_xaxis else r"$x$")
    ax.set_ylabel(r"$\arccos(x)$")
    ax.minorticks_off()
    ax.legend(loc="best")

    base = out_dir / f"arccos_actual_vs_pred_p{p}"
    fig.savefig(str(base) + ".png")
    fig.savefig(str(base) + ".eps", format="eps")

    if show_figures:
        plt.show()
    else:
        plt.close(fig)

In [3]:
if __name__ == "__main__":
    import time
    from pathlib import Path

    total_start = time.perf_counter()

    # Fixed parameters
    m = 6          # magnitude qubits
    shots = 256

    # Test range (|x| must be < 1 representably for magReg)
    xs = list(np.linspace(-0.9, 0.9, 31, endpoint=False))

    # IMPORTANT: use existing base folder "figures", and create a subfolder like "B1_arccos"
    fig_dir = Path("figures") / "B1_arccos_cordic"

    # Plot p range
    p_max = 6
    plot_ps = set(range(1, p_max + 1))

    for p in sorted(plot_ps):
        run_start = time.perf_counter()

        print("\n" + "=" * 70)
        print(f"RUN START: p = {p}")
        print("=" * 70)

        rows = test_arccos_circuit(
            m=m,
            p=p,
            xs=xs,
            shots=shots,
            clean_cordic_ancillas=True,
            seed=1234,
        )

        # Plot after table/summary prints
        plot_arccos_actual_vs_predicted(
            rows=rows,
            p=p,
            out_dir=fig_dir,              # figures/B1_arccos
            show_figures=False,
            use_xhat_on_xaxis=False,
        )

        run_end = time.perf_counter()
        run_elapsed = run_end - run_start

        print("=" * 70)
        print(f"RUN END: p = {p}")
        print(f"Run Execution time (p={p}): {run_elapsed:.6f} seconds")
        print("=" * 70)

    total_end = time.perf_counter()
    total_elapsed = total_end - total_start

    print("\n" + "-" * 70)
    print(f"Total Execution time: {total_elapsed:.6f} seconds")
    print("-" * 70)


RUN START: p = 1
┌────┬───────────┬───────────┬───────────────┬───────────────┬───────────────┬───────────────┬───────────────┐
│ #  │   x_req   │  x_quant  │   acos(xq)    │  mean(theta)  │    abs_err    │  std(theta)   │  mode(theta)  │
├────┼───────────┼───────────┼───────────────┼───────────────┼───────────────┼───────────────┼───────────────┤
│  1 │ -0.900000 │ -0.906250 │  2.7051236249 │  2.5000000000 │  0.2051236249 │  0.0000000000 │  2.5000000000 │
│  2 │ -0.841935 │ -0.843750 │  2.5750283230 │  2.5000000000 │  0.0750283230 │  0.0000000000 │  2.5000000000 │
│  3 │ -0.783871 │ -0.781250 │  2.4674621469 │  2.5000000000 │  0.0325378531 │  0.0000000000 │  2.5000000000 │
│  4 │ -0.725806 │ -0.718750 │  2.3727991046 │  2.5000000000 │  0.1272008954 │  0.0000000000 │  2.5000000000 │
│  5 │ -0.667742 │ -0.656250 │  2.2866343870 │  2.5000000000 │  0.2133656130 │  0.0000000000 │  2.5000000000 │
│  6 │ -0.609677 │ -0.625000 │  2.2459278597 │  2.5000000000 │  0.2540721403 │  0.0000000000 │

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


RUN END: p = 1
Run Execution time (p=1): 6.551549 seconds

RUN START: p = 2
┌────┬───────────┬───────────┬───────────────┬───────────────┬───────────────┬───────────────┬───────────────┐
│ #  │   x_req   │  x_quant  │   acos(xq)    │  mean(theta)  │    abs_err    │  std(theta)   │  mode(theta)  │
├────┼───────────┼───────────┼───────────────┼───────────────┼───────────────┼───────────────┼───────────────┤
│  1 │ -0.900000 │ -0.906250 │  2.7051236249 │  2.0000000000 │  0.7051236249 │  0.0000000000 │  2.0000000000 │
│  2 │ -0.841935 │ -0.843750 │  2.5750283230 │  2.0000000000 │  0.5750283230 │  0.0000000000 │  2.0000000000 │
│  3 │ -0.783871 │ -0.781250 │  2.4674621469 │  2.0000000000 │  0.4674621469 │  0.0000000000 │  2.0000000000 │
│  4 │ -0.725806 │ -0.718750 │  2.3727991046 │  2.0000000000 │  0.3727991046 │  0.0000000000 │  2.0000000000 │
│  5 │ -0.667742 │ -0.656250 │  2.2866343870 │  2.0000000000 │  0.2866343870 │  0.0000000000 │  2.0000000000 │
│  6 │ -0.609677 │ -0.625000 │  2.24

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


RUN END: p = 2
Run Execution time (p=2): 35.692607 seconds

RUN START: p = 3
┌────┬───────────┬───────────┬───────────────┬───────────────┬───────────────┬───────────────┬───────────────┐
│ #  │   x_req   │  x_quant  │   acos(xq)    │  mean(theta)  │    abs_err    │  std(theta)   │  mode(theta)  │
├────┼───────────┼───────────┼───────────────┼───────────────┼───────────────┼───────────────┼───────────────┤
│  1 │ -0.900000 │ -0.906250 │  2.7051236249 │  3.2500000000 │  0.5448763751 │  0.0000000000 │  3.2500000000 │
│  2 │ -0.841935 │ -0.843750 │  2.5750283230 │  3.2500000000 │  0.6749716770 │  0.0000000000 │  3.2500000000 │
│  3 │ -0.783871 │ -0.781250 │  2.4674621469 │  3.2500000000 │  0.7825378531 │  0.0000000000 │  3.2500000000 │
│  4 │ -0.725806 │ -0.718750 │  2.3727991046 │  2.2500000000 │  0.1227991046 │  0.0000000000 │  2.2500000000 │
│  5 │ -0.667742 │ -0.656250 │  2.2866343870 │  2.2500000000 │  0.0366343870 │  0.0000000000 │  2.2500000000 │
│  6 │ -0.609677 │ -0.625000 │  2.2

...

RUN END: p = 4
Run Execution time (p=4): 225.122018 seconds

RUN START: p = 5
┌────┬───────────┬───────────┬───────────────┬───────────────┬───────────────┬───────────────┬───────────────┐
│ #  │   x_req   │  x_quant  │   acos(xq)    │  mean(theta)  │    abs_err    │  std(theta)   │  mode(theta)  │
├────┼───────────┼───────────┼───────────────┼───────────────┼───────────────┼───────────────┼───────────────┤
│  1 │ -0.900000 │ -0.906250 │  2.7051236249 │  2.9375000000 │  0.2323763751 │  0.0000000000 │  2.9375000000 │
│  2 │ -0.841935 │ -0.843750 │  2.5750283230 │  2.5625000000 │  0.0125283230 │  0.0000000000 │  2.5625000000 │
│  3 │ -0.783871 │ -0.781250 │  2.4674621469 │  2.4375000000 │  0.0299621469 │  0.0000000000 │  2.4375000000 │
│  4 │ -0.725806 │ -0.718750 │  2.3727991046 │  2.3125000000 │  0.0602991046 │  0.0000000000 │  2.3125000000 │
│  5 │ -0.667742 │ -0.656250 │  2.2866343870 │  2.1875000000 │  0.0991343870 │  0.0000000000 │  2.1875000000 │
│  6 │ -0.609677 │ -0.625000 │  2.

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


RUN END: p = 5
Run Execution time (p=5): 410.083063 seconds

RUN START: p = 6
┌────┬───────────┬───────────┬───────────────┬───────────────┬───────────────┬───────────────┬───────────────┐
│ #  │   x_req   │  x_quant  │   acos(xq)    │  mean(theta)  │    abs_err    │  std(theta)   │  mode(theta)  │
├────┼───────────┼───────────┼───────────────┼───────────────┼───────────────┼───────────────┼───────────────┤
│  1 │ -0.900000 │ -0.906250 │  2.7051236249 │  2.7656250000 │  0.0605013751 │  0.0000000000 │  2.7656250000 │
│  2 │ -0.841935 │ -0.843750 │  2.5750283230 │  2.5156250000 │  0.0594033230 │  0.0000000000 │  2.5156250000 │
│  3 │ -0.783871 │ -0.781250 │  2.4674621469 │  2.5156250000 │  0.0481628531 │  0.0000000000 │  2.5156250000 │
│  4 │ -0.725806 │ -0.718750 │  2.3727991046 │  2.3593750000 │  0.0134241046 │  0.0000000000 │  2.3593750000 │
│  5 │ -0.667742 │ -0.656250 │  2.2866343870 │  2.2968750000 │  0.0102406130 │  0.0000000000 │  2.2968750000 │
│  6 │ -0.609677 │ -0.625000 │  2.

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


RUN END: p = 6
Run Execution time (p=6): 683.023050 seconds

----------------------------------------------------------------------
Total Execution time: 1462.416130 seconds
----------------------------------------------------------------------
